In [1]:
!pip install -q pyspark

In [2]:
from pyspark import SparkContext, SparkConf

# Configurar y arrancar el contexto de Spark (el motor de bajo nivel para RDDs)
conf = SparkConf().setAppName("DemoRDD_Clasico").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)

print("¡Spark inicializado correctamente!")
print(f"Versión de Spark: {sc.version}")

¡Spark inicializado correctamente!
Versión de Spark: 4.0.4


In [3]:
# Descargar dataset directamente desde un CSV en repositorio público GitHub, dándole un nombre (-O)
!wget -O 50_Startups.csv https://raw.githubusercontent.com/gakudo-ai/open-datasets/main/50_Startups.csv

# ALTERNATIVA PARA WINDOWS
# !curl -o 50_Startups.csv https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/50_Startups.csv

# Cargar el archivo de texto en bruto en un RDD.
# Cada elemento del RDD será una línea de texto del archivo CSV.
lineas_rdd = sc.textFile("50_Startups.csv")

# Mostrar los datos del RDD al completo
lineas_rdd.collect()

--2026-09-16 10:47:42--  https://raw.githubusercontent.com/gakudo-ai/open-datasets/main/50_Startups.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2436 (2.4K) [text/plain]
Saving to: ‘50_Startups.csv’

50_Startups.csv     100%[===================>]   2.38K  --.-KB/s    in 0s      

2026-09-16 10:47:42 (31.9 MB/s) - ‘50_Startups.csv’ saved [2436/2436]



['R&D Spend,Administration,Marketing Spend,State,Profit',
 '165349.2,136897.8,471784.1,New York,192261.83',
 '162597.7,151377.59,443898.53,California,191792.06',
 '153441.51,101145.55,407934.54,Florida,191050.39',
 '144372.41,118671.85,383199.62,New York,182901.99',
 '142107.34,91391.77,366168.42,Florida,166187.94',
 '131876.9,99814.71,362861.36,New York,156991.12',
 '134615.46,147198.87,127716.82,California,156122.51',
 '130298.13,145530.06,323876.68,Florida,155752.6',
 '120542.52,148718.95,311613.29,New York,152211.77',
 '123334.88,108679.17,304981.62,California,149759.96',
 '101913.08,110594.11,229160.95,Florida,146121.95',
 '100671.96,91790.61,249744.55,California,144259.4',
 '93863.75,127320.38,249839.44,Florida,141585.52',
 '91992.39,135495.07,252664.93,California,134307.35',
 '119943.24,156547.42,256512.92,Florida,132602.65',
 '114523.61,122616.84,261776.23,New York,129917.04',
 '78013.11,121597.55,264346.06,California,126992.93',
 '94657.16,145077.58,282574.31,New York,125370.3

In [4]:
# Demostración de Evaluación Perezosa (Lazy Evaluation):
# Esto NO ejecuta ninguna lectura pesada todavía, solo define el plan.
print("Número de particiones iniciales:", lineas_rdd.getNumPartitions())

# Acción para ver las primeras 5 líneas
print("\nPrimeras 5 líneas del RDD en bruto:")
for linea in lineas_rdd.take(5):
    print(linea)

Número de particiones iniciales: 2

Primeras 5 líneas del RDD en bruto:
R&D Spend,Administration,Marketing Spend,State,Profit
165349.2,136897.8,471784.1,New York,192261.83
162597.7,151377.59,443898.53,California,191792.06
153441.51,101145.55,407934.54,Florida,191050.39
144372.41,118671.85,383199.62,New York,182901.99


**Ejemplos de operaciones de limpieza y transformación sobre RDDs**

In [5]:
# 1. Quitar la cabecera del CSV (nombres de columnas)
cabecera = lineas_rdd.first()
filas_rdd = lineas_rdd.filter(lambda linea: linea != cabecera)

# 2. Transformar cada línea de texto separada por comas en una estructura de datos manipulable (Array/Lista)
# Mapeo (Map): Convertimos texto plano en una tupla o lista dividida por comas
datos_rdd = filas_rdd.map(lambda linea: linea.split(","))

# 3. Suponer que queremos filtrar las startups cuyo "State" (columna 3, índice 3) sea 'New York'
# (Limpiando las comillas que suelen traer los CSVs)
ny_rdd = datos_rdd.filter(lambda cols: len(cols) > 3 and "New York" in cols[3])

print(f"Número total de startups en New York: {ny_rdd.count()}")

# Visualizar un par de resultados aplicando una acción (collect)
print("\nEjemplos de startups en New York:")
for row in ny_rdd.take(2):
    print(row)

Número total de startups en New York: 17

Ejemplos de startups en New York:
['165349.2', '136897.8', '471784.1', 'New York', '192261.83']
['144372.41', '118671.85', '383199.62', 'New York', '182901.99']


In [6]:
# Cierre del contexto para liberar recursos del cluster local
sc.stop()
print("\nSesión de Spark finalizada correctamente.")


Sesión de Spark finalizada correctamente.
